# Sanity 03 — the pretrained model

Mirrors `scripts/03_pretrain.py`. Stage 03 trains the ~29M-param field-split encoder with
independent per-field masking (each dynamic field draws its own 15% mask; predicting a hidden
field from its visible siblings is the task) + merchant InfoNCE, and checkpoints every epoch.
Here: load the checkpoint, count params, run one forward, and plot the training curves.

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath(".."))   # template root (src/)
BASE = os.environ.get("FM_BASE", "/mnt/user_storage/transaction-fm-v2")
SCALE = "full"                               # flip to "xl" / "xxl" to inspect those runs

In [ ]:
ckpt = f"{BASE}/model/{SCALE}"
mcfg = json.load(open(f"{ckpt}/model_config.json"))
vocab = json.load(open(f"{ckpt}/vocab.json"))
print(json.dumps(mcfg, indent=2))
print("dynamic fields:", vocab["dynamic_fields"])
print("infonce fields:", vocab.get("infonce_fields"), "| signal fields:", vocab.get("signal_fields"))

In [ ]:
import torch
from src.model import build_model

model = build_model(f"{ckpt}/vocab.json", arch=mcfg["arch"], max_len=mcfg["max_len"])
model.load_state_dict(torch.load(f"{ckpt}/model.pt", map_location="cpu"))
model.eval()
print(f"{sum(p.numel() for p in model.parameters())/1e6:.1f}M parameters")

In [ ]:
# One real window through the model -> the 512-d state at the last position (THE embedding).
import pyarrow.dataset as pads
import torch

row = pads.dataset(f"{BASE}/tokenized/{SCALE}/eval", format="parquet").head(1).to_pandas().iloc[0]
batch = {}
for c in row.index:
    if c.startswith("s_"):
        batch[c] = torch.tensor([int(row[c])])                      # [1]
    elif c.startswith("d_") or c in ("attention_mask", "time_bucket"):
        v = np.asarray(row[c])
        dt = torch.float32 if c.endswith(("_log", "_frac")) else torch.long
        batch[c] = torch.as_tensor(v, dtype=dt)[None]               # [1, seq_len]
with torch.no_grad():
    emb = model.sequence_embedding(batch, pooling="last")
print("embedding shape:", tuple(emb.shape), "| norm:", float(emb.norm()))

In [ ]:
# Training curves from the TensorBoard event files (rank-0 writer).
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
import glob, matplotlib.pyplot as plt

runs = sorted(glob.glob(f"{BASE}/tensorboard/fm_{SCALE}_*"))
print("runs:", [os.path.basename(r) for r in runs])
acc = EventAccumulator(runs[-1]); acc.Reload()
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for tag, ax in [("train/loss", axes[0]), ("epoch/acc_macro", axes[1])]:
    if tag in acc.Tags()["scalars"]:
        ev = acc.Scalars(tag)
        ax.plot([e.step for e in ev], [e.value for e in ev]); ax.set_title(tag)
plt.tight_layout()

**What to check:** `periodic_amount`/`periodic_time` true and `intra_tx_attention` absent/false
in the config; ~29M params; MLM loss falling smoothly (full run ends ≈5); macro field accuracy
rising to ~0.75+. Per-field curves (`field_ce/*`) are in the same TB run if you want the
merchant-head story.